In [1]:
using Pkg
Pkg.activate(".")

using Dates
using Random          # <--- add this
using Printf

using Oceananigans
using Oceananigans.Units: minute, minutes, hour, hours, day, days, meter, meters, kilometer, kilometers


  Activating new project at `/gpfs/gibbs/project/eisaman/ka659/quals_ready_minor`


In [2]:
# --------------------------------------------------
# computing architecture
# --------------------------------------------------
architecture = CPU()

CPU()

In [3]:
# --------------------------------------------------
# grid
# --------------------------------------------------
const Nx = 100  # number of x points 
const Ny = 100  # number of y points 
const Nz = 1    # number of z points 
    
const Lx = 100kilometers # (m) domain x extent
const Ly = 100kilometers # (m) domain y extent
const Lz = 10meters      # (m) domain depth

# --------------------------------------------------
# initial conditions (same logic, inside loop)
# --------------------------------------------------
const u_initial = 0.0
const v_initial = 0.0
const T_initial = 20.0
const S_initial = 35.0

35.0

In [5]:
#each seed = one realization of the random-wind experiment, with its own NetCDF file

seeds = [11, 22, 33]   # pick any integers; these label your realizations


3-element Vector{Int64}:
 11
 22
 33

In [6]:
for seed in seeds

    println("=== Running realization with seed = $seed ===")
    Random.seed!(seed)   # resets the RNG for this run

    # --------------------------------------------------
    # grid
    # --------------------------------------------------
    
    grid = RectilinearGrid(
        architecture, 
        topology = (Periodic, Periodic, Bounded),
        size = (Nx, Ny, Nz),
        halo = (3,3,2),
        x = (0, Lx),
        y = (0, Ly),
        z = (-Lz, 0)
    )

    # --------------------------------------------------
    # coriolis, closure, buoyancy (same as before)
    # --------------------------------------------------
    coriolis = FPlane(f=1e-4)

    closure = AnisotropicMinimumDissipation()

    buoyancy = SeawaterBuoyancy(
        equation_of_state = LinearEquationOfState(
            thermal_expansion = 2e-4,
            haline_contraction = 8e-4)
    )

    # --------------------------------------------------
    # surface stress (still random, but now reproducible per seed)
    # --------------------------------------------------
    u₁₀ = 10    # m s⁻¹, average wind velocity 10 meters above the ocean
    ρₒ = 1026.0 # kg m⁻³
    cᴰ = 2.5e-3
    ρₐ = 1.225  # kg m⁻³

    @inline wind_stress_x(x, y, t, p) = (- ρₐ / ρₒ * cᴰ * u₁₀ * abs(u₁₀)) * rand()
    @inline wind_stress_y(x, y, t, p) = (- ρₐ / ρₒ * cᴰ * u₁₀ * abs(u₁₀)) * rand()

    u_top_bcs = FluxBoundaryCondition(wind_stress_x, parameters=(k=4π, ω=3.0, τ=1e-4))
    v_top_bcs = FluxBoundaryCondition(wind_stress_y, parameters=(k=4π, ω=3.0, τ=1e-4))

    u_bcs = FieldBoundaryConditions(top = u_top_bcs) 
    v_bcs = FieldBoundaryConditions(top = v_top_bcs)

    T_bcs = FieldBoundaryConditions()
    S_bcs = FieldBoundaryConditions()
    c_bcs = FieldBoundaryConditions()

    # --------------------------------------------------
    # instantiate model (same as before)
    # --------------------------------------------------
    model = NonhydrostaticModel(; grid, buoyancy,
                                timestepper = :QuasiAdamsBashforth2,
                                advection = WENO(),
                                tracers = (:T, :S, :c),
                                coriolis = coriolis,
                                closure = closure,
                                boundary_conditions = (u=u_bcs, v=v_bcs, T=T_bcs, S=S_bcs, c=c_bcs))

    # --------------------------------------------------
    # initial conditions (same logic, inside loop)
    # --------------------------------------------------
    
    #const u_initial = 0.0
    #const v_initial = 0.0
    #const T_initial = 20.0
    #const S_initial = 35.0

    center_x_index = Int(grid.Nx / 2)
    center_y_index = Int(grid.Ny / 2)
    surface_z_index = grid.Nz

    c_initial = model.tracers.c

    dx = Lx / Nx
    dy = Ly / Ny
    dz = Lz / Nz
    dv = dx * dy * dz

    c_initial .= 0.0
    c_initial[center_x_index, center_y_index, surface_z_index] = 1.0

    set!(model, u=u_initial, v=v_initial, T=T_initial, S=S_initial, c=c_initial)

    # --------------------------------------------------
    # simulation and outputs
    # --------------------------------------------------
    simulation = Simulation(model, Δt=10minutes, stop_time=100days)

    wizard = TimeStepWizard(cfl=0.2)
    simulation.callbacks[:wizard] = Callback(wizard, IterationInterval(10))

    progress_message(sim) = @printf("seed=%d | Iteration: %04d, time: %s, Δt: %s, max(|u|) = %.1e ms⁻¹, wall time: %s\n",
                                    seed,
                                    iteration(sim), prettytime(sim), prettytime(sim.Δt),
                                    maximum(abs, sim.model.velocities.u), prettytime(sim.run_wall_time))

    add_callback!(simulation, progress_message, IterationInterval(20))

    # ----- output writer with seed in filename -----
    base_filename = @sprintf("output-tracer-randwind-365-seed%03d-%s", seed, string(today()))

    simulation.output_writers[:surface_slice_writer] = NetCDFOutputWriter(
        model, 
        (; model.velocities.u, model.velocities.v, model.tracers.S, model.tracers.T, model.tracers.c),
        filename = base_filename * ".nc",
        schedule = AveragedTimeInterval(1hour, window=1hour),
        indices=(:, :, grid.Nz),
        overwrite_existing = true
    )

    run!(simulation)

end  # loop over seeds


=== Running realization with seed = 11 ===


[ Info: Using the advection scheme UpwindBiased(order=1) in the z-direction because size(grid, 3) = 1
[ Info: Initializing simulation...


seed=11 | Iteration: 0000, time: 0 seconds, Δt: 11 minutes, max(|u|) = 0.0e+00 ms⁻¹, wall time: 0 seconds


[ Info:     ... simulation initialization complete (18.604 seconds)
[ Info: Executing initial time step...
[ Info:     ... initial time step complete (4.924 seconds).


seed=11 | Iteration: 0020, time: 3.605 hours, Δt: 9.705 minutes, max(|u|) = 3.0e-01 ms⁻¹, wall time: 24.039 seconds
seed=11 | Iteration: 0040, time: 6.256 hours, Δt: 5.931 minutes, max(|u|) = 4.1e-01 ms⁻¹, wall time: 24.337 seconds
seed=11 | Iteration: 0060, time: 8.089 hours, Δt: 5.169 minutes, max(|u|) = 3.6e-01 ms⁻¹, wall time: 24.626 seconds
seed=11 | Iteration: 0080, time: 9.773 hours, Δt: 5.319 minutes, max(|u|) = 2.7e-01 ms⁻¹, wall time: 24.926 seconds
seed=11 | Iteration: 0100, time: 11.575 hours, Δt: 6.325 minutes, max(|u|) = 1.3e-01 ms⁻¹, wall time: 25.217 seconds
seed=11 | Iteration: 0120, time: 13.580 hours, Δt: 7.654 minutes, max(|u|) = 5.9e-02 ms⁻¹, wall time: 25.518 seconds
seed=11 | Iteration: 0140, time: 16 hours, Δt: 9.261 minutes, max(|u|) = 9.7e-02 ms⁻¹, wall time: 25.808 seconds
seed=11 | Iteration: 0160, time: 19 hours, Δt: 11.206 minutes, max(|u|) = 1.7e-01 ms⁻¹, wall time: 26.109 seconds
seed=11 | Iteration: 0180, time: 22.154 hours, Δt: 7.681 minutes, max(|u|) 

...

[ Info:     ... simulation initialization complete (17.627 ms)
[ Info: Executing initial time step...
[ Info:     ... initial time step complete (24.187 ms).


seed=22 | Iteration: 0020, time: 3.605 hours, Δt: 9.603 minutes, max(|u|) = 3.1e-01 ms⁻¹, wall time: 289.077 ms
seed=22 | Iteration: 0040, time: 6.254 hours, Δt: 5.947 minutes, max(|u|) = 4.2e-01 ms⁻¹, wall time: 594.738 ms
seed=22 | Iteration: 0060, time: 8.091 hours, Δt: 5.197 minutes, max(|u|) = 3.7e-01 ms⁻¹, wall time: 881.443 ms
seed=22 | Iteration: 0080, time: 9.778 hours, Δt: 5.331 minutes, max(|u|) = 2.6e-01 ms⁻¹, wall time: 1.170 seconds
seed=22 | Iteration: 0100, time: 11.579 hours, Δt: 6.373 minutes, max(|u|) = 1.3e-01 ms⁻¹, wall time: 1.459 seconds
seed=22 | Iteration: 0120, time: 13.701 hours, Δt: 7.712 minutes, max(|u|) = 6.5e-02 ms⁻¹, wall time: 1.756 seconds
seed=22 | Iteration: 0140, time: 16.141 hours, Δt: 9.331 minutes, max(|u|) = 9.4e-02 ms⁻¹, wall time: 2.053 seconds
seed=22 | Iteration: 0160, time: 19.171 hours, Δt: 11.291 minutes, max(|u|) = 1.8e-01 ms⁻¹, wall time: 2.356 seconds
seed=22 | Iteration: 0180, time: 22.307 hours, Δt: 7.461 minutes, max(|u|) = 3.9e-01

[ Info: Simulation is stopping after running for 7.152 minutes.
[ Info: Simulation time 100 days equals or exceeds stop time 100 days.
[ Info: Using the advection scheme UpwindBiased(order=1) in the z-direction because size(grid, 3) = 1
[ Info: Initializing simulation...
[ Info:     ... simulation initialization complete (16.188 ms)
[ Info: Executing initial time step...
[ Info:     ... initial time step complete (24.094 ms).


seed=33 | Iteration: 0020, time: 3.605 hours, Δt: 9.755 minutes, max(|u|) = 3.0e-01 ms⁻¹, wall time: 292.201 ms
seed=33 | Iteration: 0040, time: 6.251 hours, Δt: 6.026 minutes, max(|u|) = 4.1e-01 ms⁻¹, wall time: 585.818 ms
seed=33 | Iteration: 0060, time: 8.089 hours, Δt: 5.173 minutes, max(|u|) = 3.6e-01 ms⁻¹, wall time: 890.361 ms
seed=33 | Iteration: 0080, time: 9.775 hours, Δt: 5.368 minutes, max(|u|) = 2.6e-01 ms⁻¹, wall time: 1.173 seconds
seed=33 | Iteration: 0100, time: 11.579 hours, Δt: 6.371 minutes, max(|u|) = 1.3e-01 ms⁻¹, wall time: 1.468 seconds
seed=33 | Iteration: 0120, time: 13.701 hours, Δt: 7.708 minutes, max(|u|) = 6.3e-02 ms⁻¹, wall time: 1.757 seconds
seed=33 | Iteration: 0140, time: 16.141 hours, Δt: 9.327 minutes, max(|u|) = 9.9e-02 ms⁻¹, wall time: 2.056 seconds
seed=33 | Iteration: 0160, time: 19.171 hours, Δt: 11.286 minutes, max(|u|) = 1.8e-01 ms⁻¹, wall time: 2.350 seconds
seed=33 | Iteration: 0180, time: 22.297 hours, Δt: 7.578 minutes, max(|u|) = 3.9e-01

Excessive output truncated after 524429 bytes.